# Assgn_013 — Token ladder (2M → 5M → 10M → optional 50M)

Use this notebook on a **fresh** Colab runtime when you want shorter budgets before the graded 50M run.

## Steps

1. Upload `Assign_0013_colab.zip` to **`/content/`** (not `/content/content/`).
2. Run **Cell 0** (unzip + pip).
3. Run **Cell 1** (setup). Drive mount is attempted; if it fails, reports stay under `/content/Assign_0013/reports`.
4. Run **Smoke** (~4k tokens) → confirm gate + PNGs → zip/download.
5. Run **2M** → zip/download.
6. Run **5M** → zip/download.
7. Run **10M** → zip/download.
8. (Optional / graded) Run **50M** with `smoke=False` → zip.

Each `run_all(..., tokens=N)` trains **four** variants at budget **N**. Keep `smoke=True` for 2/5/10M so the 50M token gate does not false-FAIL. Do **not** set `RESUME` between different budgets (each gets a new `vNNN_...` folder).

**USD:** Colab T4 ≈ 0.12 USD/h. Free tier: set `ASSGN013_FREE_COLAB=1` in Cell 1.


## Cell 0 — unzip once

Edit the zip path if your file landed elsewhere (e.g. `/content/content/Assign_0013_colab.zip`).


In [ ]:
ZIP = '/content/Assign_0013_colab.zip'
# If upload went into a nested folder:
# ZIP = '/content/content/Assign_0013_colab.zip'

!unzip -q -o {ZIP} -d /content/Assign_0013
%cd /content/Assign_0013
!test -f .env || cp .env.example .env
!pip -q install -r requirements.txt
!pwd
!ls -la


## Cell 1 — setup + optional Drive

Drive mount often fails (popup blocked / timeout). This cell **catches** that and falls back to local `reports/`. Always run the zip cell after each budget if Drive is not mounted.


In [ ]:
from pathlib import Path
import os
import shutil
import sys

REPORTS_ROOT = None  # set below if Drive mounts

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    REPORTS_ROOT = '/content/drive/MyDrive/Assign_0013/reports'
    Path(REPORTS_ROOT).mkdir(parents=True, exist_ok=True)
    print('Drive OK → reports at', REPORTS_ROOT)
except Exception as exc:
    REPORTS_ROOT = None
    print('Drive mount failed — using local reports/ and zip-after-each.')
    print('Reason:', type(exc).__name__, str(exc)[:200])

ROOT = Path.cwd()
if not (ROOT / 'src').is_dir():
    ROOT = Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

env_path = ROOT / '.env'
example = ROOT / '.env.example'
if not env_path.is_file() and example.is_file():
    shutil.copy2(example, env_path)
    print('created .env from .env.example')

os.environ.setdefault('ASSGN013_HOST', 'colab')
os.environ.setdefault('ASSGN013_USD_PER_HOUR', '0.12')
# Free Colab (no units): uncomment next two lines
# os.environ['ASSGN013_FREE_COLAB'] = '1'
# os.environ['ASSGN013_USD_PER_HOUR'] = '0.00'

if env_path.is_file():
    lines = []
    for line in env_path.read_text(encoding='utf-8').splitlines():
        if line.startswith('ASSGN013_HOST='):
            lines.append('ASSGN013_HOST=colab')
        elif line.startswith('ASSGN013_USD_PER_HOUR=') and os.environ.get('ASSGN013_USD_PER_HOUR'):
            lines.append(f"ASSGN013_USD_PER_HOUR={os.environ['ASSGN013_USD_PER_HOUR']}")
        else:
            lines.append(line)
    env_path.write_text('\n'.join(lines) + '\n', encoding='utf-8')

if REPORTS_ROOT:
    os.environ['ASSGN013_REPORTS_ROOT'] = REPORTS_ROOT
else:
    os.environ.pop('ASSGN013_REPORTS_ROOT', None)

from IPython.display import Image, display
from src.pipeline.rev_lab import run_all, format_gate_print
from src.utils.versioning import read_latest

reports_root = Path(os.environ['ASSGN013_REPORTS_ROOT']) if os.environ.get('ASSGN013_REPORTS_ROOT') else (ROOT / 'reports')
reports_root.mkdir(parents=True, exist_ok=True)


def zip_latest_results(results: dict) -> Path:
    """Zip version folder to /content/<version>.zip for download."""
    vid = results['version']
    vdir = Path(results['reports_dir'])
    out = Path('/content') / f'{vid}.zip'
    if out.exists():
        out.unlink()
    shutil.make_archive(str(out.with_suffix('')), 'zip', vdir)
    mb = round(out.stat().st_size / 1e6, 2)
    print(f'zipped {out} ({mb} MB) — download from Files pane before disconnect')
    return out


def show_primary(results: dict) -> None:
    vdir = Path(results['reports_dir'])
    for name in (
        'assgn013_loss_trajectories.png',
        'assgn013_memory_vs_seq_batch.png',
        'assgn013_speed_cost_dashboard.png',
    ):
        p = vdir / name
        if p.is_file():
            display(Image(filename=str(p)))
        else:
            print('missing', p)


print('ROOT', ROOT)
print('reports_root', reports_root)
print('env_exists', env_path.is_file())


## Smoke (wiring check, ~4k tokens)

Run this **before** 2M. Confirms imports, GPU, OOM probe, PNGs, and zip path. Takes about a minute — not a training budget.

In [ ]:
# Tiny smoke — four runs x 4096 tokens (smoke=True exempts 50M gate)
results = run_all(reports_root, root=ROOT, tokens=4096, smoke=True)
print('\n'.join(format_gate_print(results)))
print('LATEST', read_latest(reports_root))
show_primary(results)
zip_latest_results(results)

## 2M ladder step

Four runs × 2M tokens. Then zip/download.


In [ ]:
results = run_all(reports_root, root=ROOT, tokens=2_000_000, smoke=True)
print('\n'.join(format_gate_print(results)))
print('LATEST', read_latest(reports_root))
show_primary(results)
zip_latest_results(results)


## 5M ladder step

New version folder (do not RESUME the 2M id). Four runs × 5M.


In [ ]:
results = run_all(reports_root, root=ROOT, tokens=5_000_000, smoke=True)
print('\n'.join(format_gate_print(results)))
print('LATEST', read_latest(reports_root))
show_primary(results)
zip_latest_results(results)


## 10M ladder step

Four runs × 10M. Zip/download before starting 50M.


In [ ]:
results = run_all(reports_root, root=ROOT, tokens=10_000_000, smoke=True)
print('\n'.join(format_gate_print(results)))
print('LATEST', read_latest(reports_root))
show_primary(results)
zip_latest_results(results)


## Optional: graded 50M (hours on T4)

`smoke=False` enforces tokens ≥ 50M. Leave `RESUME = None` on first start. After a disconnect, set `RESUME` to the same `vNNN_...` id **only if** that folder still exists (Drive or re-uploaded zip).


In [ ]:
RESUME = None  # e.g. 'v004_colab_t4_full' after disconnect
if RESUME:
    os.environ['ASSGN013_RUN_VERSION'] = RESUME

results = run_all(
    reports_root,
    root=ROOT,
    tokens=50_000_000,
    smoke=False,
    version=RESUME,
)
print('\n'.join(format_gate_print(results)))
print('LATEST', read_latest(reports_root))
show_primary(results)
zip_latest_results(results)


## Re-zip last results (if you skipped auto-zip)


In [ ]:
# Re-run after any ladder cell if download did not happen
zip_latest_results(results)
